In [ ]:
from pathlib import Path
import seaborn as sns

PROJECT_ROOT = Path("/home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese")

GCN_CSV = PROJECT_ROOT / "notebooks/GNN/results/GCN_4Classes_MultiSubject/all_subjects_results_gcn.csv"
GAT_CSV = PROJECT_ROOT / "notebooks/GNN/results/GNN_GAT_4Classes_MultiSubject/all_subjects_results_gat.csv"
OUT_DIR = PROJECT_ROOT / "images/GCN_vs_GAT"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("GCN CSV:", GCN_CSV)
print("GAT CSV:", GAT_CSV)
print("Output folder:", OUT_DIR)
print("GCN exists:", GCN_CSV.exists())
print("GAT exists:", GAT_CSV.exists())

sns.set_theme(style="whitegrid", context="talk")

In [ ]:
import numpy as np
import pandas as pd

gcn_df = pd.read_csv(GCN_CSV)[["subject", "mean_balanced_accuracy"]].rename(
    columns={"mean_balanced_accuracy": "gcn_bal_acc"}
)
gat_df = pd.read_csv(GAT_CSV)[["subject", "mean_balanced_accuracy"]].rename(
    columns={"mean_balanced_accuracy": "gat_bal_acc"}
)

merged = gcn_df.merge(gat_df, on="subject", how="inner", validate="one_to_one")

merged["gcn_bal_acc_pct"] = merged["gcn_bal_acc"] * 100.0
merged["gat_bal_acc_pct"] = merged["gat_bal_acc"] * 100.0
merged["delta_gat_minus_gcn"] = merged["gat_bal_acc"] - merged["gcn_bal_acc"]
merged["delta_gat_minus_gcn_pct"] = merged["delta_gat_minus_gcn"] * 100.0

eps = 1e-12
merged["winner"] = np.select(
    [merged["delta_gat_minus_gcn"] > eps, merged["delta_gat_minus_gcn"] < -eps],
    ["GAT", "GCN"],
    default="Tie",
)

print("Subjects in merged table:", len(merged))
display(merged.sort_values("subject").reset_index(drop=True))

In [ ]:
GCN_RESULTS_DIR = PROJECT_ROOT / "notebooks/GNN/results/GCN_4Classes_MultiSubject"
GAT_RESULTS_DIR = PROJECT_ROOT / "notebooks/GNN/results/GNN_GAT_4Classes_MultiSubject"

In [ ]:
def load_fold_balanced_accuracy(base_dir, model_name):
    rows = []
    for csv_path in sorted(base_dir.glob("sub-*/fold_balanced_accuracy.csv")):
        df = pd.read_csv(csv_path)
        needed = {"subject", "fold", "balanced_accuracy"}
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
        df = df[["subject", "fold", "balanced_accuracy"]].copy()
        df["model"] = model_name
        rows.append(df)
    if not rows:
        return pd.DataFrame(columns=["subject", "fold", "balanced_accuracy", "model"])
    out = pd.concat(rows, ignore_index=True)
    out["fold"] = out["fold"].astype(int)
    out["balanced_accuracy"] = out["balanced_accuracy"].astype(float)
    return out

In [ ]:
gcn_fold_df = load_fold_balanced_accuracy(GCN_RESULTS_DIR, "GCN")
gat_fold_df = load_fold_balanced_accuracy(GAT_RESULTS_DIR, "GAT")

if gcn_fold_df.empty or gat_fold_df.empty:
    raise FileNotFoundError("No fold_balanced_accuracy.csv files found. Rerun the training notebooks.")

In [ ]:
shared_subjects = sorted(set(gcn_fold_df["subject"]) & set(gat_fold_df["subject"]))
if not shared_subjects:
    raise ValueError("No overlapping subjects with fold-level files between GCN and GAT.")

In [ ]:
fold_plot_df = pd.concat(
    [gcn_fold_df[gcn_fold_df["subject"].isin(shared_subjects)],
     gat_fold_df[gat_fold_df["subject"].isin(shared_subjects)]],
    ignore_index=True,
)
fold_plot_df["balanced_accuracy_pct"] = fold_plot_df["balanced_accuracy"] * 100.0
fold_plot_df["model"] = pd.Categorical(fold_plot_df["model"], categories=["GCN", "GAT"], ordered=True)
fold_plot_df = fold_plot_df.sort_values(["subject", "model", "fold"]).reset_index(drop=True)
fold_plot_df.to_csv(OUT_DIR / "per_fold_balanced_accuracy_long.csv", index=False)

check_counts = fold_plot_df.groupby(["subject", "model"])["fold"].nunique().reset_index(name="n_folds")
bad = check_counts[check_counts["n_folds"] != 4]
if not bad.empty:
    print("Warning: some subject/model pairs do not have exactly 4 folds:")
    display(bad)

display(fold_plot_df.head(12))
print("Shared subjects:", shared_subjects)
print("Rows in fold dataframe:", len(fold_plot_df))
print("Expected:", len(shared_subjects) * 4 * 2)

In [ ]:
from matplotlib import pyplot as plt

palette = {"GCN": "#4C78A8", "GAT": "#72B7B2"}

fold_plot_df_plot = fold_plot_df.copy()
fold_plot_df_plot["_group"] = "All subjects"

fig, ax = plt.subplots(figsize=(5, 6), dpi=150)
sns.violinplot(
    data=fold_plot_df_plot,
    x="_group", y="balanced_accuracy_pct", hue="model",
    split=True, palette=palette, inner="quart", cut=0, linewidth=1.2, ax=ax,
)
ax.axhline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Fold Balanced Accuracy: GCN vs GNN\n({len(shared_subjects)} subjects x 4 folds each)")
ax.legend(title="Model", loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig(OUT_DIR / "violin_fold_balanced_accuracy_gcn_vs_gat.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
per_subject_table = merged[
    ["subject", "gcn_bal_acc_pct", "gat_bal_acc_pct", "delta_gat_minus_gcn_pct", "winner"]
].copy().sort_values("delta_gat_minus_gcn_pct", ascending=False).reset_index(drop=True)

per_subject_table_rounded = per_subject_table.copy()
per_subject_table_rounded[["gcn_bal_acc_pct", "gat_bal_acc_pct", "delta_gat_minus_gcn_pct"]] = \
    per_subject_table_rounded[["gcn_bal_acc_pct", "gat_bal_acc_pct", "delta_gat_minus_gcn_pct"]].round(2)

per_subject_table_rounded.to_csv(OUT_DIR / "per_subject_balanced_accuracy_table_rounded.csv", index=False)
display(per_subject_table_rounded)

print("GNN wins:", int((per_subject_table["winner"] == "GAT").sum()))
print("GCN wins:", int((per_subject_table["winner"] == "GCN").sum()))
print("Ties:", int((per_subject_table["winner"] == "Tie").sum()))
print("Mean delta (GNN - GCN):", round(per_subject_table["delta_gat_minus_gcn_pct"].mean(), 2), "pp")

In [ ]:
import pingouin as pg

x = merged["gcn_bal_acc"].to_numpy(dtype=float)
y = merged["gat_bal_acc"].to_numpy(dtype=float)
d = y - x

normality_df = pg.normality(d)
ttest_df = pg.ttest(y, x, paired=True)
wilcoxon_df = pg.wilcoxon(y, x, alternative="two-sided")

normality_df.to_csv(OUT_DIR / "stats_normality_diff.csv", index=False)
ttest_df.to_csv(    OUT_DIR / "stats_paired_ttest.csv", index=False)
wilcoxon_df.to_csv( OUT_DIR / "stats_wilcoxon.csv", index=False)

In [ ]:
def pick_value(df, candidates, default=np.nan):
    if df.empty:
        return float(default)
    for c in candidates:
        if c in df.columns:
            try:
                return float(df[c].iloc[0])
            except Exception:
                return float(default)
    return float(default)

In [ ]:
paired_t_p = pick_value(ttest_df, ["p-val", "p_val", "pvalue", "p"])
paired_t_d = pick_value(ttest_df, ["cohen-d", "cohen_d", "cohend", "d"])
wilcoxon_p = pick_value(wilcoxon_df, ["p-val", "p_val", "pvalue", "p"])

summary_df = pd.DataFrame([{
    "n_subjects": len(merged),
    "gcn_mean_bal_acc": float(np.mean(x)),
    "gcn_std_bal_acc": float(np.std(x, ddof=1)),
    "gat_mean_bal_acc": float(np.mean(y)),
    "gat_std_bal_acc": float(np.std(y, ddof=1)),
    "mean_diff_gat_minus_gcn": float(np.mean(d)),
    "mean_diff_percent_points": float(np.mean(d) * 100.0),
    "paired_t_p": paired_t_p,
    "paired_t_cohen_d": paired_t_d,
    "wilcoxon_p": wilcoxon_p,
}])
summary_df.to_csv(OUT_DIR / "balanced_accuracy_stats_summary.csv", index=False)

In [ ]:
print("Normality test of paired differences:")
display(normality_df)
print("Paired t-test:")
display(ttest_df)
print("Wilcoxon signed-rank:")
display(wilcoxon_df)
print("Compact summary:")
display(summary_df)

In [ ]:
plot_df = merged.sort_values("delta_gat_minus_gcn_pct", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 0.6 * len(plot_df) + 2), dpi=150)
y_pos = np.arange(len(plot_df))

for i, row in plot_df.iterrows():
    ax.plot([row["gcn_bal_acc_pct"], row["gat_bal_acc_pct"]], [i, i],
            color="gray", alpha=0.7, linewidth=2)

ax.scatter(plot_df["gcn_bal_acc_pct"], y_pos, s=70, color="#4C78A8", label="GCN", zorder=3)
ax.scatter(plot_df["gat_bal_acc_pct"], y_pos, s=70, color="#72B7B2", label="GAT", zorder=3)
ax.axvline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["subject"])
ax.set_xlabel("Balanced Accuracy (%)")
ax.set_title("Per-subject Balanced Accuracy: GCN vs GAT")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_dumbbell_gcn_vs_gat.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta_df = merged.sort_values("delta_gat_minus_gcn_pct", ascending=False).reset_index(drop=True)
colors = np.where(delta_df["delta_gat_minus_gcn_pct"] >= 0, "#2E8B57", "#B22222")

fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
bars = ax.bar(delta_df["subject"], delta_df["delta_gat_minus_gcn_pct"], color=colors, alpha=0.9)

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta (GNN - GCN), percentage points")
ax.set_title("Per-subject Change in Balanced Accuracy (GNN - GCN)")
ax.tick_params(axis="x", rotation=45)

for b, val in zip(bars, delta_df["delta_gat_minus_gcn_pct"]):
    ax.text(
        b.get_x() + b.get_width() / 2,
        val + (0.4 if val >= 0 else -0.6),
        f"{val:.1f}", ha="center",
        va="bottom" if val >= 0 else "top", fontsize=9,
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_delta_gat_minus_gcn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
lines = ["Per-subject GCN vs GNN (Balanced Accuracy)", ""]

for _, r in per_subject_table_rounded.iterrows():
    lines.append(
        f"{r['subject']}: GCN={r['gcn_bal_acc_pct']:.2f}%, "
        f"GAT={r['gat_bal_acc_pct']:.2f}%, "
        f"Delta={r['delta_gat_minus_gcn_pct']:+.2f} pp, "
        f"Winner={r['winner']}"
    )

lines += [
    "",
    f"GNN wins: {int((per_subject_table['winner'] == 'GAT').sum())}",
    f"GCN wins: {int((per_subject_table['winner'] == 'GCN').sum())}",
    f"Ties: {int((per_subject_table['winner'] == 'Tie').sum())}",
    f"Mean Delta (GNN-GCN): {per_subject_table['delta_gat_minus_gcn_pct'].mean():+.2f} pp",
    f"Paired t-test p: {summary_df.loc[0, 'paired_t_p']:.6g}",
    f"Wilcoxon p: {summary_df.loc[0, 'wilcoxon_p']:.6g}",
]

report_path = OUT_DIR / "per_subject_report.txt"
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", report_path)
print("\n".join(lines[:8]))

In [ ]:
print("Saved files in:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)